# 10 · Heaps and Top-K

After this notebook you can use `heapq` fluently (min-heaps, max-heaps, tuple priorities), recognise the "top k / k-th / merge / running median / scheduling" signals, solve the classic heap problems, and explain why a vector search ends with `np.argpartition` rather than a full sort.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [Trees and BST](09_trees_and_bst.ipynb), [Sorting algorithms](07_sorting_algorithms.ipynb)

## Why this matters in interviews

- The words "k largest", "k closest", "k most frequent", "merge k sorted", "running median" and "schedule by priority" all mean **heap**. Saying "a min-heap of size k, O(n log k)" before writing code is half the answer.
- You must know what `heapq` gives you (a **min**-heap on a plain list) and the two standard tricks: negate for a max-heap, and `(priority, counter, item)` tuples.
- Every retrieval system ends in a top-k over similarity scores, and a sharded one merges per-shard top-k lists. Interviewers for AI-engineer roles like to hear the cost of that step and how to make it cheap.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ve03` | Your vector database has 10 million documents. How does it find the nearest without comparing all 10 million? (the exact-search baseline and its cost) |
| `ve22` | How would you shard a vector index across machines? (scatter-gather and merging per-shard top-k) |
| `fo15` | What are top-k and top-p, and which should you use? (both filters implemented) |

## What interviewers ask

- "Find the k-th largest element" (LeetCode 215) — and "can you do it in O(n)?"
- "Top k frequent elements" (347), "k closest points to the origin" (973).
- "Merge k sorted lists" (23).
- "Find the median from a data stream" (295).
- "Task scheduler with a cooldown" (621); "minimum meeting rooms" (253, full treatment in [notebook 13](13_greedy_and_intervals.ipynb)).
- Follow-ups: "why a **min**-heap for the k **largest**?", "what if the data does not fit in memory?", "what is the complexity of `heapify`?"

In [ ]:
import bisect
import heapq
import itertools
import math
import random
import statistics
import time
from collections import Counter, deque

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.titlesize": 11,
                     "legend.frameon": False})
BLUE, ORANGE, AQUA, YELLOW, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#0b0b0b", "#52514e"

def is_heap(a):
    """Min-heap invariant: every parent <= its children."""
    return all(a[(i - 1) // 2] <= a[i] for i in range(1, len(a)))

## 1. The pattern

**In plain English:** a heap is a priority line. You can always see the smallest item instantly, and adding or removing an item costs only a handful of swaps (log n). It does **not** keep everything sorted — only "every parent is smaller than its children" — and that weaker promise is exactly why it is cheaper than sorting when you only want the best few.

**How it is stored:** a complete binary tree packed into a list. The node at index `i` has children `2i + 1` and `2i + 2` and parent `(i - 1) // 2`. No pointers, and the smallest item is always `heap[0]`.

In [ ]:
def draw_heap(a, title):
    n, depth = len(a), int(math.log2(len(a))) + 1
    width = 2 ** (depth - 1)
    pos = {}
    for i in range(n):
        d = int(math.log2(i + 1)); j = i - (2 ** d - 1)
        pos[i] = ((j + 0.5) * width / 2 ** d, -d)
    fig, ax = plt.subplots(figsize=(9, 4))
    for i in range(1, n):
        p = (i - 1) // 2
        ax.plot([pos[p][0], pos[i][0]], [pos[p][1], pos[i][1]], color="#b5b3ac", lw=1.5, zorder=1)
    for i, (x, y) in pos.items():
        ax.scatter([x], [y], s=520, zorder=2, color="#fde7dc" if i == 0 else "#e7f0ff",
                   edgecolors=ORANGE if i == 0 else BLUE, linewidths=1.5)
        ax.text(x, y, str(a[i]), ha="center", va="center", zorder=3)
        ax.text(x + 0.2, y + 0.22, f"[{i}]", fontsize=8, color=MUTED)
    y0, cell = -depth - 0.3, width / n
    for i, v in enumerate(a):
        ax.add_patch(plt.Rectangle((i * cell, y0 - 0.3), cell * 0.94, 0.6,
                                   fc="#fde7dc" if i == 0 else "#e7f0ff", ec=ORANGE if i == 0 else BLUE))
        ax.text(i * cell + cell * 0.47, y0, str(v), ha="center", va="center")
        ax.text(i * cell + cell * 0.47, y0 - 0.55, str(i), ha="center", fontsize=8, color=MUTED)
    ax.text(-0.15, y0, "as a list:", ha="right", va="center", color=MUTED)
    ax.set_xlim(-1.3, width + 0.2); ax.set_ylim(y0 - 0.8, 0.5); ax.axis("off")
    ax.set_title(title)
    plt.show()

nums = [9, 4, 7, 1, 8, 2, 6, 3, 5, 0]
heap = nums[:]
heapq.heapify(heap)                    # in place, O(n), returns None
assert is_heap(heap) and heap[0] == min(nums)
draw_heap(heap, "heapify([9, 4, 7, 1, 8, 2, 6, 3, 5, 0]): parent <= children; the minimum sits at index 0")

**The top-k template** (the one to write from memory): to keep the k **largest** items, keep a **min**-heap of size k. Its root is the weakest of your current top k, so each new item only has to beat `heap[0]`.

```python
heap = []
for x in stream:
    if len(heap) < k:
        heapq.heappush(heap, x)
    elif x > heap[0]:
        heapq.heapreplace(heap, x)   # pop the weakest, push x: one O(log k) operation
# heap now holds the k largest; heap[0] is the k-th largest
```
O(n log k) time, **O(k) memory** — it works on a stream you cannot hold in memory.

## 2. `heapq` in five minutes

| Operation | Call | Cost |
|---|---|---|
| build a heap from a list | `heapq.heapify(a)` | O(n), in place |
| peek at the minimum | `a[0]` | O(1) |
| push | `heapq.heappush(a, x)` | O(log n) |
| pop the minimum | `heapq.heappop(a)` | O(log n) |
| push, then pop (cheaper than both) | `heapq.heappushpop(a, x)` | O(log n) |
| pop, then push | `heapq.heapreplace(a, x)` | O(log n) |
| k smallest / largest of any iterable | `heapq.nsmallest(k, it, key=...)` / `nlargest` | O(n log k) |
| merge already-sorted iterables, lazily | `heapq.merge(*its, key=..., reverse=...)` | O(N log k) |

In [ ]:
h = []
for x in [5, 3, 8, 1]:
    heapq.heappush(h, x)
print("after pushes      :", h, "| min =", h[0])
print("heappushpop(h, 0) :", heapq.heappushpop(h, 0), "<- 0 is pushed and immediately popped")
print("heapreplace(h, 0) :", heapq.heapreplace(h, 0), "<- the old min is popped first, then 0 goes in")
print("pop everything    :", [heapq.heappop(h) for _ in range(len(h))])
print("nlargest(3)       :", heapq.nlargest(3, nums), "| nsmallest(2):", heapq.nsmallest(2, nums))
print("longest 2 words   :", heapq.nlargest(2, ["kiwi", "banana", "fig", "cherry"], key=len))
print("heapify returns   :", heapq.heapify([3, 1, 2]))

rnd = random.Random(0)
for _ in range(200):
    data = [rnd.randint(-50, 50) for _ in range(rnd.randint(0, 40))]
    hp = data[:]
    heapq.heapify(hp)
    assert is_heap(hp) and [heapq.heappop(hp) for _ in range(len(hp))] == sorted(data)
print("heapify + repeated heappop == sorted(), on 200 random lists")

### Max-heaps and priorities

`heapq` only does **min**-heaps (Python 3.14 added `heappush_max` and friends; this venv is 3.12). Two standard tricks:

1. **Negate** numbers: push `-x`, and negate again when you pop.
2. Push **tuples** `(priority, tie_breaker, item)`. Tuples compare element by element, so equal priorities fall through to the next field. If that next field is a dict or an object with no ordering, Python raises `TypeError` — a counter from `itertools.count()` makes every tuple unique, and keeps equal priorities in insertion (FIFO) order.

In [ ]:
max_heap = []
for x in nums:
    heapq.heappush(max_heap, -x)
largest_three = [-heapq.heappop(max_heap) for _ in range(3)]
print("max-heap via negation, three largest:", largest_three)
assert largest_three == sorted(nums, reverse=True)[:3]

try:
    bad = []
    heapq.heappush(bad, (1, {"name": "a"}))
    heapq.heappush(bad, (1, {"name": "b"}))          # tie on priority -> compares the dicts
    raise AssertionError("expected TypeError")
except TypeError as e:
    print("tie on priority with dict payloads -> TypeError:", e)

pq, counter = [], itertools.count()
for priority, name in [(2, "write tests"), (1, "fix prod bug"), (3, "refactor"), (1, "answer page")]:
    heapq.heappush(pq, (priority, next(counter), {"name": name}))
order = [heapq.heappop(pq)[2]["name"] for _ in range(len(pq))]
print("with a counter tie-breaker:", order)
assert order == ["fix prod bug", "answer page", "write tests", "refactor"]

## 3. Classic heap problems

### Problem 1 — K-th largest element (LeetCode 215)

`[3, 2, 1, 5, 6, 4], k = 2` → 5. **Brute force:** sort descending and index: O(n log n). **Heap:** the top-k template; the root is the answer: O(n log k), O(k) memory. **Selection (quickselect):** O(n) on average — in NumPy that is `np.partition`.

In [ ]:
def kth_largest_sort(nums, k):
    return sorted(nums, reverse=True)[k - 1]

def kth_largest_heap(nums, k):
    heap = []
    for x in nums:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)
    return heap[0]

def kth_largest_partition(nums, k):
    return int(np.partition(np.asarray(nums), len(nums) - k)[len(nums) - k])

assert kth_largest_heap([3, 2, 1, 5, 6, 4], 2) == 5
assert kth_largest_heap([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4          # duplicates count separately
for _ in range(300):
    data = [rnd.randint(-100, 100) for _ in range(rnd.randint(1, 60))]
    k = rnd.randint(1, len(data))
    assert kth_largest_heap(data, k) == kth_largest_sort(data, k) == kth_largest_partition(data, k)
print("heap, sort and np.partition agree on 300 random cases (k from 1 to n)")

### Problem 2 — Top k frequent elements (LeetCode 347)

`[1, 1, 1, 2, 2, 3], k = 2` → `[1, 2]`. Count with a `Counter`, then take the k largest counts: O(n log k). Fun fact worth saying: `Counter.most_common(k)` is implemented with `heapq.nlargest`. **O(n) alternative:** bucket sort — bucket `c` holds the values that occur `c` times; read buckets from the top.

In [ ]:
def top_k_frequent(nums, k):
    counts = Counter(nums)
    return heapq.nlargest(k, counts, key=counts.get)

def top_k_frequent_bucket(nums, k):
    counts = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for x, c in counts.items():
        buckets[c].append(x)
    out = []
    for c in range(len(nums), 0, -1):
        for x in buckets[c]:
            out.append(x)
            if len(out) == k:
                return out
    return out

assert top_k_frequent([1, 1, 1, 2, 2, 3], 2) == [1, 2] == top_k_frequent_bucket([1, 1, 1, 2, 2, 3], 2)
assert top_k_frequent([7], 1) == [7]
for _ in range(200):
    data = [rnd.randint(0, 15) for _ in range(rnd.randint(1, 80))]
    counts = Counter(data)
    k = rnd.randint(1, len(counts))
    best = sorted(counts.values(), reverse=True)[:k]           # ties may pick different values,
    for fn in (top_k_frequent, top_k_frequent_bucket):          # but the counts must be the top k
        assert sorted((counts[x] for x in fn(data, k)), reverse=True) == best
print(top_k_frequent(list("mississippi river"), 3), "<- three most frequent characters")

### Problem 3 — K closest points to the origin (LeetCode 973)

Keep a **max**-heap of size k keyed on distance (negate it), so the root is the farthest of your current k and a closer point replaces it. Compare **squared** distances — no `sqrt` needed to order them.

In [ ]:
def k_closest(points, k):
    heap = []                                    # (-squared distance, x, y): a max-heap by distance
    for x, y in points:
        d = x * x + y * y
        if len(heap) < k:
            heapq.heappush(heap, (-d, x, y))
        elif d < -heap[0][0]:
            heapq.heapreplace(heap, (-d, x, y))
    return [(x, y) for _, x, y in heap]

assert k_closest([(1, 3), (-2, 2)], 1) == [(-2, 2)]
assert sorted(k_closest([(3, 3), (5, -1), (-2, 4)], 2)) == [(-2, 4), (3, 3)]
for _ in range(200):
    pts = [(rnd.randint(-20, 20), rnd.randint(-20, 20)) for _ in range(rnd.randint(1, 50))]
    k = rnd.randint(1, len(pts))
    dist = lambda p: p[0] ** 2 + p[1] ** 2
    assert sorted(map(dist, k_closest(pts, k))) == sorted(map(dist, pts))[:k]

prng = np.random.default_rng(0)
cloud = [tuple(p) for p in prng.uniform(-10, 10, size=(200, 2)).round(2)]
near = set(k_closest(cloud, 15))
radius = max(math.hypot(x, y) for x, y in near)
fig, ax = plt.subplots(figsize=(6.4, 4.5))
ax.scatter(*zip(*[p for p in cloud if p not in near]), s=12, color="#b5b3ac", label="other points")
ax.scatter(*zip(*near), s=30, color=ORANGE, label="15 closest")
ax.add_patch(plt.Circle((0, 0), radius, fill=False, ec=ORANGE, lw=1.5))
ax.scatter([0], [0], marker="+", s=120, color=INK)
ax.set_aspect("equal"); ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
ax.set_title("k closest points: a size-15 max-heap on distance")
plt.show()

**Complexity:** O(n log k) time, O(k) space. (Quickselect would be O(n) average.)

### Problem 4 — Merge k sorted lists (LeetCode 23)

**Brute force:** concatenate everything and sort: O(N log N) for N total items. **Heap:** keep one "front" item per list in a heap of size k; pop the smallest, push that list's next item: O(N log k). The tuple is `(value, list_index, position)` — the list index also breaks ties, which matters in the linked-list version because `ListNode` objects cannot be compared. `heapq.merge` is the lazy built-in version.

In [ ]:
def merge_k_sorted(lists):
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists) if lst]
    heapq.heapify(heap)
    out = []
    while heap:
        value, i, j = heapq.heappop(heap)
        out.append(value)
        if j + 1 < len(lists[i]):
            heapq.heappush(heap, (lists[i][j + 1], i, j + 1))
    return out

example_lists = [[1, 4, 5], [1, 3, 4], [2, 6]]
print(merge_k_sorted(example_lists))
assert merge_k_sorted(example_lists) == [1, 1, 2, 3, 4, 4, 5, 6]
assert merge_k_sorted([]) == [] and merge_k_sorted([[]]) == []
for _ in range(200):
    lists = [sorted(rnd.randint(0, 50) for _ in range(rnd.randint(0, 8))) for _ in range(rnd.randint(0, 6))]
    expected = sorted(itertools.chain.from_iterable(lists))
    assert merge_k_sorted(lists) == expected == list(heapq.merge(*lists))
print("heap merge == sort-everything == heapq.merge on 200 random inputs")

### GenAI tie-in: merging results from a sharded vector index (`ve22`)

With **scatter-gather** sharding, every shard searches its own slice and returns its local top-k sorted by score; the router merges them — that is merge-k-sorted-lists, and only the first k merged items are needed, so the lazy `heapq.merge` stops early.

The interview detail: **each shard must return the full k**, not about `k / shards`. The global top 10 is rarely spread evenly across shards; if one shard holds four of them but returns only three, one is gone. Let us measure that over 200 queries.

In [ ]:
srng = np.random.default_rng(0)
n_docs, n_shards, k = 20_000, 4, 10
shard_of = srng.integers(0, n_shards, size=n_docs)              # which shard stores each document
shard_ids = [np.flatnonzero(shard_of == s) for s in range(n_shards)]

def shard_search(scores, ids, top):
    """One shard's local answer: its `top` best (score, doc_id), best first."""
    best = ids[np.argsort(-scores[ids])[:top]]
    return [(float(scores[i]), int(i)) for i in best]

def scatter_gather(scores, top_per_shard):
    per_shard = [shard_search(scores, ids, top_per_shard) for ids in shard_ids]
    return list(itertools.islice(heapq.merge(*per_shard, reverse=True), k))   # lazy: stops after k

cheap_k = math.ceil(k / n_shards)
recalls = []
for query in range(200):
    scores_q = srng.normal(size=n_docs)                         # similarity of every doc to this query
    exact = [(float(scores_q[i]), int(i)) for i in np.argsort(-scores_q)[:k]]
    assert scatter_gather(scores_q, k) == exact                 # full k per shard: always exact
    recalls.append(len(set(scatter_gather(scores_q, cheap_k)) & set(exact)) / k)
    if query == 0:
        print("query 0 - how many of the true top-10 each shard holds:",
              dict(sorted(Counter(int(shard_of[i]) for _, i in exact).items())))
recalls = np.array(recalls)
print(f"each shard returns k = {k}: recall@10 = 1.000 on all 200 queries")
print(f"each shard returns {cheap_k}:  mean recall@10 = {recalls.mean():.3f}; "
      f"{(recalls < 1).mean():.0%} of queries lose at least one true top-10 document")
assert recalls.mean() < 1

### Problem 5 — Median of a data stream (LeetCode 295)

Numbers arrive one at a time; report the median at any moment.

**Brute force:** keep a sorted list (`bisect.insort`) and read the middle — O(n) per insert because the list shifts. **Two heaps:** a **max**-heap `low` holds the smaller half and a **min**-heap `high` the larger half, with `len(low) == len(high)` or one more. The median is `low`'s top (or the average of both tops): O(log n) per insert, O(1) per query.

```
      low (max-heap)          high (min-heap)
   [ ... smaller half ]  |  [ larger half ... ]
                  top ^  |  ^ top      median = top of low, or the mean of both tops
```

In [ ]:
class MedianFinder:
    def __init__(self):
        self.low, self.high = [], []          # low stores negatives (max-heap)

    def add(self, x):
        heapq.heappush(self.low, -x)
        heapq.heappush(self.high, -heapq.heappop(self.low))      # largest of low moves up
        if len(self.high) > len(self.low):                        # rebalance: low may have one extra
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):
            return -self.low[0]
        return (-self.low[0] + self.high[0]) / 2

mf, sorted_so_far = MedianFinder(), []
for x in [rnd.randint(-1000, 1000) for _ in range(2000)]:
    mf.add(x)
    bisect.insort(sorted_so_far, x)                   # the brute-force reference
    assert mf.median() == statistics.median(sorted_so_far)
    assert len(mf.low) - len(mf.high) in (0, 1)
print("two-heap median == statistics.median after every one of 2,000 inserts")

**GenAI tie-in:** a running **p50 latency** of LLM calls is exactly this. Latencies are skewed and a few requests time out; the mean jumps with every timeout, the median barely moves. (Production monitoring usually uses streaming sketches for p95/p99, but the two-heap median is the interview answer.)

In [ ]:
lrng = np.random.default_rng(0)
latency = lrng.lognormal(mean=0.0, sigma=0.5, size=1500)       # seconds, typical LLM call spread
timeouts = lrng.random(1500) < 0.03
latency[timeouts] = 30.0                                        # 3% of calls hit a 30 s timeout

mf, running_median, running_mean, total = MedianFinder(), [], [], 0.0
for i, x in enumerate(latency, start=1):
    mf.add(float(x)); total += x
    running_median.append(mf.median()); running_mean.append(total / i)

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(range(len(latency)), latency, s=4, color="#b5b3ac", label="individual calls")
ax.plot(running_mean, color=BLUE, lw=2, label="running mean")
ax.plot(running_median, color=ORANGE, lw=2, label="running median (two heaps)")
ax.set_yscale("log"); ax.set_ylim(0.1, 400); ax.legend(loc="upper center", ncol=3)
ax.set_xlabel("request number"); ax.set_ylabel("latency in seconds (log scale)")
ax.set_title("3% timeouts drag the mean up; the median stays at the typical latency")
plt.show()
print(f"final mean {running_mean[-1]:.2f} s vs final median {running_median[-1]:.2f} s")
assert running_mean[-1] > 1.5 * running_median[-1]

### Problem 6 — Task scheduler (LeetCode 621), a preview of greedy + heap

Tasks `A A A B B B`, cooldown `n = 2` between two runs of the same task, one task (or idle) per time unit. Minimum total time? **Greedy with a max-heap:** at each tick run the available task with the most copies left, and park it in a cooldown queue until it is allowed again. There is also an O(n) counting formula: `max(len(tasks), (max_count - 1) * (n + 1) + number_of_tasks_with_max_count)`; the simulation checks it.

In [ ]:
def schedule_tasks(tasks, n):
    heap = [(-c, t) for t, c in Counter(tasks).items()]      # max-heap on remaining count
    heapq.heapify(heap)
    schedule, cooling = [], deque()                          # cooling: (ready_time, -count, task)
    while heap or cooling:
        now = len(schedule)
        if cooling and cooling[0][0] <= now:
            _, c, task = cooling.popleft()
            heapq.heappush(heap, (c, task))
        if heap:
            c, task = heapq.heappop(heap)
            schedule.append(task)
            if c + 1 < 0:
                cooling.append((now + n + 1, c + 1, task))
        else:
            schedule.append("idle")
    return schedule

def least_interval_formula(tasks, n):
    counts = Counter(tasks).values()
    top = max(counts)
    return max(len(tasks), (top - 1) * (n + 1) + sum(c == top for c in counts))

print(" ".join(schedule_tasks(list("AAABBB"), 2)))
assert len(schedule_tasks(list("AAABBB"), 2)) == 8 and len(schedule_tasks(list("AAABBB"), 0)) == 6
assert len(schedule_tasks(list("AAAAAABCDEFG"), 2)) == 16
for _ in range(300):
    tasks = [rnd.choice("ABCDE") for _ in range(rnd.randint(1, 25))]
    n = rnd.randint(0, 4)
    assert len(schedule_tasks(tasks, n)) == least_interval_formula(tasks, n)
print("heap simulation == counting formula on 300 random cases")

**Meeting rooms II (LeetCode 253), preview:** sort meetings by start and keep a min-heap of end times of the rooms in use; if the earliest-ending room is free, reuse it (`heapreplace`), otherwise open a new one. The heap size at the end is the answer. [Notebook 13](13_greedy_and_intervals.ipynb) covers it properly, with plots and the LLM-concurrency connection.

In [ ]:
def min_meeting_rooms(intervals):
    ends = []
    for start, end in sorted(intervals):
        if ends and ends[0] <= start:
            heapq.heapreplace(ends, end)       # the room that frees up first is free: reuse it
        else:
            heapq.heappush(ends, end)          # every room is busy: open a new one
    return len(ends)

assert min_meeting_rooms([(0, 30), (5, 10), (15, 20)]) == 2
assert min_meeting_rooms([(7, 10), (2, 4)]) == 1 and min_meeting_rooms([]) == 0
print("rooms needed for [(0,30), (5,10), (15,20)]:", min_meeting_rooms([(0, 30), (5, 10), (15, 20)]))

## 4. GenAI tie-in: top-k retrieval over similarity scores

A brute-force (exact) vector search is two steps: **score** every document (`X @ q`, O(n·d)), then **select** the k best. For the selection step you have three choices:

| Method | Complexity | Notes |
|---|---|---|
| `heapq.nlargest(k, ...)` | O(n log k) | pure Python loop; but O(k) memory and works on a stream |
| `np.argsort(-scores)[:k]` | O(n log n) | sorts all n just to keep 10 |
| `np.argpartition(-scores, k)[:k]`, then sort those k | O(n + k log k) | selection in C; the standard answer |

Let us time them on 100,000 unit vectors (64 dimensions) and one query, k = 10.

In [ ]:
vrng = np.random.default_rng(0)
n, d, k = 100_000, 64, 10
X = vrng.normal(size=(n, d)).astype(np.float32)
X /= np.linalg.norm(X, axis=1, keepdims=True)                 # unit vectors: dot product = cosine
q = vrng.normal(size=d).astype(np.float32)
q /= np.linalg.norm(q)
scores = X @ q

def topk_heapq(scores, k):
    return [i for i, _ in heapq.nlargest(k, enumerate(scores.tolist()), key=lambda t: t[1])]

def topk_sort(scores, k):
    return np.argsort(-scores)[:k].tolist()

def topk_argpartition(scores, k):
    idx = np.argpartition(-scores, k)[:k]                     # the k best, in arbitrary order: O(n)
    return idx[np.argsort(-scores[idx])].tolist()             # sort only those k

assert topk_heapq(scores, k) == topk_sort(scores, k) == topk_argpartition(scores, k)

def bench_ms(fn, repeat=7):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return 1000 * statistics.median(times)

timings = {"score all docs: X @ q": bench_ms(lambda: X @ q),
           "heapq.nlargest (Python)": bench_ms(lambda: topk_heapq(scores, k)),
           "np.argsort (full sort)": bench_ms(lambda: topk_sort(scores, k)),
           "np.argpartition + sort k": bench_ms(lambda: topk_argpartition(scores, k))}
for name, ms in timings.items():
    print(f"{name:<26} {ms:7.2f} ms")
assert 3 * timings["np.argpartition + sort k"] < timings["np.argsort (full sort)"]     # "several times faster"
assert timings["np.argpartition + sort k"] < timings["heapq.nlargest (Python)"]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
names = list(timings)
colors = ["#b5b3ac", BLUE, BLUE, ORANGE]
ax.barh(names, [timings[m] for m in names], color=colors, height=0.55)
for i, m in enumerate(names):
    ax.text(timings[m], i, f"  {timings[m]:.2f} ms", va="center", fontsize=9, color=MUTED)
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("median time in milliseconds (7 runs)")
ax.set_title("Top-10 of 100,000 cosine scores: argpartition selects fastest (grey = scoring cost)")
ax.set_xlim(0, max(timings.values()) * 1.25)
plt.show()

Read the chart: `argpartition` wins the selection step by a wide margin, and the full sort pays for ordering 99,990 items nobody will read. The pure-Python heap loses here only because of per-item interpreter overhead — its O(k) memory is what makes it the right tool when scores arrive in batches or as a stream (merge each batch's top-k into one small heap).

**Back-of-envelope for `ve03`:** scoring cost grows with n × d. Scale the measured scoring time from 100,000 × 64 to 10 million × 768 (a realistic embedding size):

In [ ]:
scale = (10_000_000 / n) * (768 / d)
est_ms = timings["score all docs: X @ q"] * scale
print(f"measured: {n:,} x {d} dims scored in {timings['score all docs: X @ q']:.2f} ms")
print(f"estimate: 10,000,000 x 768 dims -> about {est_ms:,.0f} ms per query on this CPU "
      f"(and {10_000_000 * 768 * 4 / 1e9:.1f} GB of float32 vectors to stream through memory)")

That is why a 10M-document store does not score everything per query: it filters by metadata first and uses an approximate index (HNSW, IVF) that only scores a few thousand candidates. The top-k selection you just timed still runs at the end — over the candidates. See [vector search with FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb).

### Same selection problem inside decoding: top-k vs top-p (`fo15`)

**Top-k sampling** keeps the k highest logits — the same selection problem, solved with `np.argpartition` in O(V) for a vocabulary of V tokens. **Top-p (nucleus)** keeps the smallest set of tokens whose probability adds up to p, which needs the tokens in sorted order. The interview point: top-k keeps the same number of tokens whether the model is sure or not; top-p adapts.

In [ ]:
V = 50_000
trng = np.random.default_rng(1)
confident = trng.normal(size=V); confident[123] += 18      # one token clearly dominates
unsure = trng.normal(size=V) * 0.5                         # almost flat distribution

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def top_k_keep(logits, k):
    return np.argpartition(-logits, k)[:k]                 # indices of the k largest logits

def top_p_keep(logits, p):
    order = np.argsort(-logits)
    cum = np.cumsum(softmax(logits)[order])
    return order[: min(int(np.searchsorted(cum, p)) + 1, len(order))]   # smallest prefix with mass >= p

kept = {}
for name, logits in [("confident", confident), ("unsure", unsure)]:
    kept[name] = (len(top_k_keep(logits, 40)), len(top_p_keep(logits, 0.9)))
    print(f"{name:>9} step: top-k=40 keeps {kept[name][0]:>2} tokens | top-p=0.9 keeps {kept[name][1]:>6,} tokens")
assert kept["confident"][0] == kept["unsure"][0] == 40
assert kept["confident"][1] == 1 and kept["unsure"][1] > 10_000
assert softmax(confident)[top_p_keep(confident, 0.9)].sum() >= 0.9

## Try it yourself

**1. K-th largest element in a stream (LeetCode 703).** Design a class whose `add(x)` returns the current k-th largest. Example: `k = 3`, start with `[4, 5, 8, 2]`; `add(3) → 4`, `add(5) → 5`, `add(10) → 5`, `add(9) → 8`, `add(4) → 8`.

In [ ]:
# Solution — try it yourself first, then run this
class KthLargest:
    def __init__(self, k, nums):
        self.k, self.heap = k, []
        for x in nums:
            self.add(x)

    def add(self, x):
        heapq.heappush(self.heap, x)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)          # keep only the k largest seen so far
        return self.heap[0]

kl = KthLargest(3, [4, 5, 8, 2])
answers = [kl.add(x) for x in [3, 5, 10, 9, 4]]
print(answers)
assert answers == [4, 5, 5, 8, 8]

**2. Sort a nearly sorted array.** Every element is at most `k` positions away from its sorted position. Sort it in O(n log k). Hint: the smallest remaining element is always among the next `k + 1`.

In [ ]:
# Solution — try it yourself first, then run this
def sort_k_sorted(a, k):
    heap, out = a[:k + 1], []
    heapq.heapify(heap)
    for x in a[k + 1:]:
        out.append(heapq.heappushpop(heap, x))
    while heap:
        out.append(heapq.heappop(heap))
    return out

for _ in range(200):
    k = rnd.randint(0, 5)
    a = sorted(rnd.randint(0, 99) for _ in range(rnd.randint(0, 40)))
    for i in range(0, len(a), k + 1):              # shuffle inside windows: nobody moves more than k
        window = a[i:i + k + 1]; rnd.shuffle(window); a[i:i + k + 1] = window
    assert sort_k_sorted(a, k) == sorted(a)
print(sort_k_sorted([2, 1, 3, 5, 4, 6, 8, 7], 1))

**3. Reorganize string (LeetCode 767).** Rearrange `s` so no two adjacent characters are equal, or return `""` if impossible. Hint: always place the most frequent character that is not the one you just placed (max-heap, hold the last one back for one turn). It is impossible exactly when some character occurs more than `(len(s) + 1) // 2` times.

In [ ]:
# Solution — try it yourself first, then run this
def reorganize(s):
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(heap)
    out, held = [], None                               # held: the character placed last, waiting a turn
    while heap:
        c, ch = heapq.heappop(heap)
        out.append(ch)
        if held:
            heapq.heappush(heap, held)
        held = (c + 1, ch) if c + 1 < 0 else None
    return "".join(out) if len(out) == len(s) else ""

assert reorganize("aab") == "aba" and reorganize("aaab") == ""
for _ in range(300):
    s = "".join(rnd.choice("abc") for _ in range(rnd.randint(1, 12)))
    r = reorganize(s)
    possible = max(Counter(s).values()) <= (len(s) + 1) // 2
    assert (r != "") == possible
    if r:
        assert Counter(r) == Counter(s) and all(a != b for a, b in zip(r, r[1:]))
print(reorganize("aaabbc"))

## Say it in an interview

**30-second answer:** "A heap is a complete binary tree in an array where every parent is at most its children, so the minimum is at index 0. Push and pop are O(log n), peek is O(1), and heapify is O(n). For the k largest of n items I keep a min-heap of size k and replace the root whenever something beats it: O(n log k) time and O(k) memory, and it works on a stream. If I have everything in a NumPy array, `argpartition` does the selection in O(n)."

**Key numbers (from this notebook's run):** for the top 10 of 100,000 scores, `argpartition` is several times faster than a full `argsort` and far faster than a Python heap loop. Scoring 10M × 768 float32 vectors exactly means streaming ~31 GB through memory per query: seconds on a laptop, and still roughly 0.1–0.3 s on a server with 100–200 GB/s of memory bandwidth — which is why vector databases filter first and use ANN indexes.

**Follow-ups they will ask:**
- *"k-th largest in O(n)?"* — quickselect (average O(n), worst O(n²)); `np.partition`.
- *"Why a min-heap for the largest?"* — the root is the weakest of the current top k, the only one a newcomer has to beat.
- *"Sharded search?"* — every shard returns the full k, merge with a heap; latency is the slowest shard's.
- *"Delete an arbitrary item?"* — `heapq` cannot find it in O(log n); use lazy deletion (mark and skip on pop) or an indexed heap.
- *"Running median?"* — two heaps, sizes balanced within one.

**Traps:** `heapq` is min-only · `heapify` returns `None` · tuples with uncomparable payloads need a tie-breaker · mutating an item inside the heap breaks the invariant · `nlargest` with `k` close to `n` is slower than `sorted` (the docs say so too).

## Pattern cheat sheet

| Signal | Heap recipe | Time / space |
|---|---|---|
| k largest / k-th largest | min-heap of size k, replace root if bigger | O(n log k) / O(k) |
| k smallest / k closest | max-heap of size k (negate keys) | O(n log k) / O(k) |
| k most frequent | `Counter` + `nlargest(k, counts, key=counts.get)` (or buckets, O(n)) | O(n log k) / O(n) |
| merge k sorted lists / shard results | heap of one front item per list | O(N log k) / O(k) |
| running median | max-heap low half + min-heap high half | O(log n) add, O(1) median |
| always process the most urgent / most frequent next | priority queue of `(priority, counter, item)` | O(log n) per op |
| rooms / servers in use over time | min-heap of end times | O(n log n) / O(n) |
| everything in a NumPy array | `np.argpartition` then sort the k | O(n + k log k) |

## Next

- [11 · Graphs](11_graphs_bfs_dfs_topo_dijkstra.ipynb) — Dijkstra is BFS with a heap.
- [13 · Greedy and intervals](13_greedy_and_intervals.ipynb) — meeting rooms with a heap, and peak LLM concurrency.
- [Vector search with FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb) · [Semantic search service](../17_interview_coding_rounds/04_semantic_search_service.ipynb)
- [Decoding: temperature, top-k, top-p](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb) · [Latency percentiles p50/p95 and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb)
- Theory: [DSA basics course — heaps chapter](../../dsa_basics/index.html) · [interview bank](../../ai_interview_prep/index.html)